# Sorting

Notes and runnable examples on sorting — anchored by **Timsort**, the adaptive hybrid that powers Python's `sorted()` and `list.sort()`. We'll see why O(n log n) is the floor for comparison sorts, what makes Timsort special, how `key=` works under the hood, then implement the full roster of classic sorts and race them against it.

**Contents**

1. **The comparison-sort floor** — why O(n log n)
2. **Timsort** — what Python actually does
3. **`key=`** — decorate-sort-undecorate
4. **The O(n²) sorts** — bubble, selection, insertion
5. **The O(n log n) sorts** — merge, quicksort, heapsort
6. **Beating the floor** — counting & radix (non-comparison)
7. **Bucket sort** — when the input is uniform
8. **When to use what**
9. **Complexity cheat-sheet**

## 1. The comparison-sort floor — why O(n log n)

Any sort that works only by **comparing pairs of elements** has a hard lower bound of **O(n log n)**. The argument is information-theoretic: there are `n!` possible orderings of the input, each comparison yields one bit (`≤` or `>`), and you need enough bits to single out the right ordering among all `n!` — so at least `log₂(n!) ≈ n log₂ n` comparisons. No comparison sort beats that in the worst case; the best you can do is hit it with a small constant and **adapt** to data that's already partly ordered.

(Sorts that *don't* compare — counting sort, radix sort — can reach O(n) for integers in a bounded range by using values directly as array indices. We'll build them in section 6.)

In [1]:
import math

print(f"{'n':>9}{'log2(n!)':>14}{'n*log2(n)':>14}")
for n in (10, 1000, 1_000_000):
    log2_factorial = math.lgamma(n + 1) / math.log(2)   # lgamma avoids building a huge factorial
    print(f"{n:>9}{log2_factorial:>14.0f}{n*math.log2(n):>14.0f}")


        n      log2(n!)     n*log2(n)
       10            22            33
     1000          8529          9966
  1000000      18488885      19931569


## 2. Timsort — what Python actually does

`list.sort()` and `sorted()` use **Timsort**, written by Tim Peters for CPython in 2002 (and since adopted by Java, V8, Android, Rust, and Swift). It's a **stable, adaptive** hybrid of merge sort and insertion sort, built on one observation: **real data is rarely random** — it arrives in already-ordered streaks. Timsort finds and exploits them:

- It scans for **runs** — maximal ascending (or strictly descending, which it flips in place) stretches — instead of assuming chaos.
- Short runs are extended to a minimum length (**minrun**, ~32–64) using **binary insertion sort**, which is excellent on tiny arrays.
- Runs are then **merged** like merge sort, with **galloping mode**: when one run keeps winning, it binary-searches ahead instead of stepping one element at a time.

The measurable payoffs: **O(n) on already-sorted (or reverse-sorted) data**, O(n log n) worst case, and **stable** (equal elements keep their original order).

In [2]:
import timeit, random
random.seed(0)

N = 1_000_000
rand     = [random.random() for _ in range(N)]
ordered  = sorted(rand)
reversed_ = ordered[::-1]

for label, data in [("random", rand), ("already sorted", ordered), ("reverse sorted", reversed_)]:
    t = timeit.timeit(lambda: sorted(data), number=3)
    print(f"  {label:15}: {t*1000:7.1f} ms")


  random         :   629.4 ms
  already sorted :    66.6 ms
  reverse sorted :    77.8 ms


Already-sorted data is several times faster — Timsort spots the single giant run and runs essentially linearly. Reverse-sorted is nearly as quick: it detects the descending run and flips it in O(n). Only random data pays the full n log n. Now the stability guarantee:

In [3]:
records = [("apple", 2), ("banana", 1), ("cherry", 2), ("date", 1), ("elder", 1)]
by_number = sorted(records, key=lambda r: r[1])
print("sorted by the number:", by_number)
print("ties keep input order -> banana, date, elder (all 1) stay in their original sequence")


sorted by the number: [('banana', 1), ('date', 1), ('elder', 1), ('apple', 2), ('cherry', 2)]
ties keep input order -> banana, date, elder (all 1) stay in their original sequence


## 3. `key=` — decorate-sort-undecorate

You sort by a derived value with `key=`, a function applied to each element. The detail that matters: Python calls `key` **exactly once per element** (n times), caches the results, sorts *those*, then reorders the originals to match. This is the **decorate–sort–undecorate (DSU)** pattern, done in C. So an expensive key is fine — it runs n times, never the O(n log n) you'd get from calling it inside every comparison.

In [4]:
import random
from operator import itemgetter

calls = 0
def slow_key(x):
    global calls
    calls += 1
    return x % 7                 # pretend this is expensive

data = list(range(1000)); random.shuffle(data)
sorted(data, key=slow_key)
print(f"key called {calls} times for {len(data)} elements  (once each, not n log n)")

# common key idioms:
people = [("Ada", 36), ("Alan", 41), ("Grace", 36)]
print("by age, then name:", sorted(people, key=itemgetter(1, 0)))   # tuple key; stable on ties
print("descending       :", sorted([3, 1, 2], reverse=True))


key called 1000 times for 1000 elements  (once each, not n log n)
by age, then name: [('Ada', 36), ('Grace', 36), ('Alan', 41)]
descending       : [3, 2, 1]


## 4. The O(n²) sorts — bubble, selection, insertion

The three quadratic sorts every course starts with. They share the same big-O but differ in the details that matter:

- **Bubble** — repeatedly swap adjacent out-of-order pairs. Its one redeeming feature: a `swapped` flag lets it **stop early**, giving O(n) on already-sorted input. Otherwise the worst of the three.
- **Selection** — each pass finds the minimum of the rest and swaps it into place. Not adaptive (always O(n²) comparisons), but does the **fewest writes**: at most n−1 swaps. Useful when a write is far more expensive than a compare.
- **Insertion** — grow a sorted prefix, inserting each new element back into place. **Adaptive** (O(n) on nearly-sorted data) and stable — which is exactly why Timsort uses it on small runs.

In [5]:
def bubble_sort(a):
    a = a[:]
    n = len(a)
    for i in range(n):
        swapped = False
        for j in range(n - 1 - i):
            if a[j] > a[j + 1]:
                a[j], a[j + 1] = a[j + 1], a[j]
                swapped = True
        if not swapped:                  # nothing moved -> already sorted, stop (O(n) best case)
            return a
    return a

def selection_sort(a):
    a = a[:]
    n = len(a)
    for i in range(n):
        lo = i
        for j in range(i + 1, n):
            if a[j] < a[lo]:
                lo = j
        a[i], a[lo] = a[lo], a[i]         # at most n-1 swaps total
    return a

def insertion_sort(a):
    a = a[:]
    for i in range(1, len(a)):
        k, j = a[i], i - 1
        while j >= 0 and a[j] > k:
            a[j + 1] = a[j]; j -= 1
        a[j + 1] = k
    return a

import random
random.seed(0)
data = [random.random() for _ in range(2000)]
ref = sorted(data)
for fn in (bubble_sort, selection_sort, insertion_sort):
    print(f"  {fn.__name__:16} correct = {fn(data) == ref}")

# the distinguishing traits, measured:
def bubble_passes(a):
    a = a[:]; n = len(a); passes = 0
    for i in range(n):
        passes += 1; swapped = False
        for j in range(n - 1 - i):
            if a[j] > a[j + 1]: a[j], a[j + 1] = a[j + 1], a[j]; swapped = True
        if not swapped: break
    return passes

def selection_swaps(a):
    a = a[:]; n = len(a); swaps = 0
    for i in range(n):
        lo = i
        for j in range(i + 1, n):
            if a[j] < a[lo]: lo = j
        if lo != i: a[i], a[lo] = a[lo], a[i]; swaps += 1
    return swaps

print()
print("bubble passes  - random:", bubble_passes(data),
      "| already-sorted:", bubble_passes(sorted(data)), "(early-exit)")
print("selection swaps:", selection_swaps(data),
      f"(<= n-1 = {len(data)-1}; bubble could do up to n(n-1)/2 = {len(data)*(len(data)-1)//2})")


  bubble_sort      correct = True
  selection_sort   correct = True
  insertion_sort   correct = True



bubble passes  - random: 1972 | already-sorted: 1 (early-exit)


selection swaps: 1993 (<= n-1 = 1999; bubble could do up to n(n-1)/2 = 1999000)


## 5. The O(n log n) sorts — merge, quicksort, heapsort

The three workhorses, each with a different trade-off:

- **Merge sort** — split, sort halves, merge. Rock-solid O(n log n) and **stable**, but needs O(n) scratch space. (Timsort is a souped-up merge sort.)
- **Quicksort** — partition around a pivot, recurse. O(n log n) average and effectively in-place, but a bad pivot gives **O(n²)** worst case, and it's not stable.
- **Heapsort** — build a heap, then pop the min repeatedly. Guaranteed O(n log n) and O(1) extra space, but not stable and cache-unfriendly. This is just "drain the heap" from the heaps notebook.

Raced against the C Timsort, the pure-Python versions are far slower — a reminder that the implementation's **constant factor** matters as much as the asymptotics.

In [6]:
import heapq, timeit, random

def merge_sort(a):
    if len(a) <= 1:
        return a
    mid = len(a) // 2
    left, right = merge_sort(a[:mid]), merge_sort(a[mid:])
    out, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:          # <= keeps it stable
            out.append(left[i]); i += 1
        else:
            out.append(right[j]); j += 1
    out.extend(left[i:]); out.extend(right[j:])
    return out

def quicksort(a):
    if len(a) <= 1:
        return a
    pivot = a[len(a) // 2]
    return (quicksort([x for x in a if x < pivot])
            + [x for x in a if x == pivot]
            + quicksort([x for x in a if x > pivot]))

def heapsort(a):                         # = the heaps notebook's "drain the heap"
    h = a[:]
    heapq.heapify(h)
    return [heapq.heappop(h) for _ in range(len(h))]

random.seed(0)
data = [random.random() for _ in range(20_000)]
ref = sorted(data)
for fn, name in [(merge_sort, "merge      O(n log n)"),
                 (quicksort,  "quick      O(n log n) avg"),
                 (heapsort,   "heap       O(n log n)"),
                 (sorted,     "Timsort    (C builtin)")]:
    t = timeit.timeit(lambda: fn(data), number=1)
    print(f"  {name:26}{t*1000:8.2f} ms   correct={fn(data) == ref}")


  merge      O(n log n)        20.80 ms   correct=True
  quick      O(n log n) avg    18.14 ms   correct=True
  heap       O(n log n)         4.22 ms   correct=True
  Timsort    (C builtin)        2.22 ms   correct=True


## 6. Beating the floor — counting & radix (non-comparison)

The O(n log n) bound only applies to sorts that **compare** elements. If your keys are integers in a bounded range, you can sort by *using the values themselves as indices* and skip comparisons entirely:

- **Counting sort** — tally how many times each value occurs, then emit them in order. **O(n + k)** for values in a range of size k. One linear pass, zero comparisons.
- **Radix sort** — counting-sort digit by digit (least-significant first), **O(d·n)** for d-digit keys. Handles large ranges that would make plain counting sort's tally array enormous.

The precondition is the whole catch: these only work on integers (or fixed-width keys), not arbitrary comparables.

In [7]:
import timeit, random

def counting_sort(a):
    if not a:
        return []
    lo, hi = min(a), max(a)
    counts = [0] * (hi - lo + 1)
    for x in a:
        counts[x - lo] += 1              # one linear pass, no comparisons
    out = []
    for value, c in enumerate(counts):
        out.extend([value + lo] * c)
    return out

def radix_sort(a):                       # LSD, base 10, non-negative ints
    if not a:
        return []
    a = a[:]
    exp = 1
    while max(a) // exp > 0:
        buckets = [[] for _ in range(10)]
        for x in a:
            buckets[(x // exp) % 10].append(x)
        a = [x for bucket in buckets for x in bucket]
        exp *= 10
    return a

random.seed(0)
low = [random.randint(0, 1000) for _ in range(1_000_000)]   # 1M ints, only ~1000 distinct
ref = sorted(low)
print("counting correct:", counting_sort(low) == ref, "| radix correct:", radix_sort(low) == ref)
print()
for fn, name in [(counting_sort, "counting_sort (pure Python)"),
                 (radix_sort,    "radix_sort    (pure Python)"),
                 (sorted,        "sorted        (C Timsort)")]:
    t = timeit.timeit(lambda: fn(low), number=1)
    print(f"  {name:30}{t*1000:7.1f} ms")


counting correct: True | radix correct: True

  counting_sort (pure Python)      57.4 ms


  radix_sort    (pure Python)     419.9 ms
  sorted        (C Timsort)       127.4 ms


On this low-cardinality data, **pure-Python counting sort beats the C Timsort** — its single linear pass does less total work than n log n comparisons, and that asymptotic edge shows through even across the Python/C divide. Radix loses *here* (its per-digit list-of-lists bookkeeping is heavy in pure Python), but both are O(n) / O(d·n) and would dominate in a compiled language or via vectorized numpy. The price is generality: there's no non-comparison sort for arbitrary objects.

## 7. Bucket sort — when the input is uniform

Counting and radix beat the comparison floor by exploiting *integer* keys. **Bucket sort** beats it by exploiting the input's *distribution*: scatter the n items into ~n buckets by value range, sort each bucket, then concatenate. When the input is **uniformly distributed**, each bucket holds ~1 item on average, so the per-bucket sorts are trivial and the whole thing runs in **O(n) expected** time.

- Split the value range into ~n equal-width buckets; item `x` goes to bucket `⌊n·(x−lo)/span⌋`.
- Sort each bucket (insertion sort shines here — buckets are tiny and nearly-sorted) and concatenate left to right. Bucket order *is* sort order, so no final merge is needed.
- The win is a probabilistic argument: under a uniform input the expected items per bucket is constant, so the total per-bucket work is O(n). It's the natural fit for **uniformly distributed floats** — the one case counting/radix can't touch directly.

The catch is the mirror image of its strength: if the data clusters, the buckets do too. **Worst case is O(n²)** — adversarial input dumps everything into one bucket, degrading to a single big insertion sort.

In [8]:
import random

def insertion_sort(a):                   # tiny buckets -> insertion sort is ideal (see section 4)
    a = a[:]
    for i in range(1, len(a)):
        k, j = a[i], i - 1
        while j >= 0 and a[j] > k:
            a[j + 1] = a[j]; j -= 1
        a[j + 1] = k
    return a

def bucket_sort(a):                      # numeric input, any range
    n = len(a)
    if n == 0:
        return []
    lo, hi = min(a), max(a)
    if hi == lo:                         # all equal -> already sorted
        return a[:]
    span = hi - lo
    buckets = [[] for _ in range(n)]     # ~n buckets, one per 1/n slice of the range
    for x in a:
        idx = int(n * (x - lo) / span)
        if idx == n:                     # x == hi lands one past the end; clamp it back
            idx -= 1
        buckets[idx].append(x)
    out = []
    for b in buckets:                    # bucket order IS sort order -> just concatenate
        out.extend(insertion_sort(b))
    return out

random.seed(0)
floats = [random.random() for _ in range(100_000)]          # uniform in [0, 1)
ints   = [random.randint(0, 10**6) for _ in range(100_000)] # uniform ints
print("uniform floats correct:", bucket_sort(floats) == sorted(floats))
print("uniform ints   correct:", bucket_sort(ints)   == sorted(ints))


uniform floats correct: True
uniform ints   correct: True


In [9]:
import timeit, random

# Why O(n) expected on uniform data: measure the bucket-size distribution.
def bucket_sizes(a):
    n = len(a); counts = [0] * n
    lo, hi = min(a), max(a); span = hi - lo
    for x in a:
        idx = int(n * (x - lo) / span); counts[min(idx, n - 1)] += 1
    return counts

random.seed(0)
uniform = [random.random() for _ in range(100_000)]
sizes = bucket_sizes(uniform)
print(f"uniform input  -> avg bucket = {sum(sizes)/len(sizes):.2f}, "
      f"max bucket = {max(sizes)}  (≈1 each -> per-bucket work is O(1))")

# Worst case: cluster n-1 items into a pinhead range, one outlier stretches the
# span -> the whole cluster maps to bucket 0 -> one giant insertion sort.
random.seed(1)
clustered = [random.random() * 1e-9 for _ in range(1999)] + [1.0]
csizes = bucket_sizes(clustered)
print(f"clustered input-> max bucket = {max(csizes)} of n = {len(clustered)}  "
      f"(one giant insertion sort -> O(n²))")
print("clustered still correct:", bucket_sort(clustered) == sorted(clustered))

# Race the pure-Python bucket sort against C Timsort on uniform floats (its home turf).
random.seed(0)
data = [random.random() for _ in range(200_000)]
for fn, name in [(bucket_sort, "bucket_sort (pure Python)"),
                 (sorted,      "sorted      (C Timsort)")]:
    t = timeit.timeit(lambda: fn(data), number=1)
    print(f"  {name:28}{t*1000:7.1f} ms")

uniform input  -> avg bucket = 1.00, max bucket = 7  (≈1 each -> per-bucket work is O(1))
clustered input-> max bucket = 1999 of n = 2000  (one giant insertion sort -> O(n²))
clustered still correct: True
  bucket_sort (pure Python)      74.3 ms


  sorted      (C Timsort)        30.1 ms


The bucket sizes tell the whole story: on uniform input nearly every bucket holds one item, so the per-bucket insertion sorts cost O(1) apiece and the total is O(n) expected. Clustered input collapses into a single bucket and the O(n²) worst case bites. The pure-Python version loses to the C Timsort on wall-clock (constant factors again — same lesson as the merge/quick/heap race), but the *algorithm* is linear-expected and a vectorized or compiled implementation would pull ahead.

Step back and the last two sections share one idea: **counting**, **radix**, and **bucket** sort all sidestep the comparison floor by exploiting structure the comparison sorts ignore — bounded integer keys for the first two, a known value distribution for bucket sort. That structure is exactly what buys you O(n); lose it (arbitrary objects, adversarial clustering) and you're back to O(n log n) or worse.

## 8. When to use what

| You need... | Reach for | Why |
|---|---|---|
| Sort a list in place | `list.sort()` | Timsort, no copy; mutates the list |
| A sorted copy of any iterable | `sorted(iterable)` | Timsort into a new list |
| Sort by a derived field | `sorted(..., key=...)` | `key` called once per element (DSU) |
| A *stable* sort | `sorted` / `list.sort` | Timsort is stable by guarantee |
| The k smallest / largest, k ≪ n | `heapq.nsmallest` / `nlargest` | O(n log k), skips a full sort |
| Keep a list sorted as you insert | `bisect.insort` | O(log n) to place, O(n) to shift |
| Sort bounded-range integers, fastest | counting / radix sort | O(n), bypasses the comparison floor |
| Sort uniformly distributed floats | bucket sort | O(n) expected; scatter, sort buckets, concatenate |

**You almost never hand-write a sort in Python** — `sorted`/`list.sort` win on nearly everything. The implementations above are for understanding, not production.

## 9. Complexity cheat-sheet

| Algorithm | Best | Average | Worst | Space | Stable | In-place | Note |
|---|:---:|:---:|:---:|:---:|:---:|:---:|---|
| **Timsort** (Python) | O(n) | O(n log n) | O(n log n) | O(n) | ✅ | ✗ | the builtin |
| Bubble | O(n) | O(n²) | O(n²) | O(1) | ✅ | ✅ | early-exit ⇒ O(n) best |
| Selection | O(n²) | O(n²) | O(n²) | O(1) | ✗ | ✅ | ≤ n−1 swaps (fewest writes) |
| Insertion | O(n) | O(n²) | O(n²) | O(1) | ✅ | ✅ | great on tiny / nearly-sorted |
| Shell | O(n log n) | ~O(n^1.3) | O(n²) | O(1) | ✗ | ✅ | gapped insertion |
| Merge | O(n log n) | O(n log n) | O(n log n) | O(n) | ✅ | ✗ | needs scratch space |
| Quicksort | O(n log n) | O(n log n) | O(n²) | O(log n) | ✗ | ✅ | bad pivot ⇒ O(n²) |
| Heapsort | O(n log n) | O(n log n) | O(n log n) | O(1) | ✗ | ✅ | drain a heap |
| Counting | O(n+k) | O(n+k) | O(n+k) | O(n+k) | ✅ | ✗ | k = value range |
| Radix | O(d·n) | O(d·n) | O(d·n) | O(n+b) | ✅ | ✗ | d digits, base b |
| Bucket | O(n+k) | O(n) | O(n²) | O(n) | ✅ | ✗ | k buckets; O(n) expected on uniform input |

<sub>Shell sort is gapped insertion sort (`shell_sort` below). `list.sort()` reorders in place, but Timsort's merge uses up to O(n) temporary space, so it isn't constant-space. k = value range; d = number of digits; b = radix base.</sub>

In [10]:
# Shell sort - insertion sort with diminishing gaps (sub-quadratic in practice)
def shell_sort(a):
    a = a[:]
    n = len(a)
    gap = n // 2
    while gap > 0:
        for i in range(gap, n):
            tmp, j = a[i], i
            while j >= gap and a[j - gap] > tmp:
                a[j] = a[j - gap]; j -= gap
            a[j] = tmp
        gap //= 2
    return a

import random
random.seed(0)
sample = [random.random() for _ in range(2000)]
print("shell_sort correct:", shell_sort(sample) == sorted(sample))


shell_sort correct: True
